# Update the Manila data (new readings -> combined dataset)
Run this **before** `transfer_learning.ipynb` and `training_cells.ipynb`. Preprocessing itself (windows, scalers, spatial split) is already a step inside those two notebooks; what they do **not** do is turn the raw sensor files into `data/Combined Dataset/<ARM>`. That is this notebook.

| Step | What it does |
|---|---|
| 1 | Settings, and what the raw files hold now (the last reading of every network) |
| 2 | Back up the current combined folders (the results so far were made from them) |
| 3 | Run `preprocessing.combine_dataset` (cleaning rules + year ranges); set `RUN = True` |
| 4 | Check: same stations as before, the new last date, the rows gained |
| 5 | Tell you what to redo, and whether the test stations moved |

Put the new raw files in `data/OpenAQ/` and `data/QC Data Converted/` first (same file names and layout as the old ones; a longer file replaces the shorter one).

## 1. Settings and what the raw files hold

In [1]:
from pathlib import Path
import glob, json, shutil, subprocess, sys
import pandas as pd

ARMS = ["from_2023_with_preinstall"]      # the combined folders to compare and back up (combine_dataset rewrites all six; these are the ones you use)
RUN = True                               # step 3 runs only when True (it rewrites data/Combined Dataset/*)
ONLY_CLEAN = False                        # True = skip the *_with_preinstall folders (not recommended: your results use them)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model" / "run_pipeline.py").exists())
DATA, MODEL = REPO / "data", REPO / "model"
COMBINED = DATA / "Combined Dataset"

def last_reading(folder, pattern):
    out = []
    for f in glob.glob(str(DATA / folder / pattern)):
        try:
            out.append(pd.read_csv(f, usecols=["local_timestamp"])["local_timestamp"].iloc[-1])
        except Exception:
            pass
    return (max(out) if out else None), len(out)

for name, folder, pat in [("OpenAQ", "OpenAQ", "openaq_*.csv"), ("QC (Quezon City)", "QC Data Converted", "qcdata_*.csv")]:
    last, n = last_reading(folder, pat)
    print(f"{name:18s} {n:3d} files, last reading: {last}")
print("(a network that stops earlier than the other is normal: the Quezon City files may end before OpenAQ; both feed the same dataset)")

OpenAQ              46 files, last reading: 2026-11-18T19:00:00+08:00
QC (Quezon City)    40 files, last reading: 2025-12-31T23:00:00+08:00
(a network that stops earlier than the other is normal: the Quezon City files may end before OpenAQ; both feed the same dataset)


## 2. Back up the current combined folders

In [2]:
stamp = pd.Timestamp.now().strftime("%Y%m%d")
for arm in ARMS:
    src, dst = COMBINED / arm, COMBINED / "_backup" / f"{arm}_{stamp}"
    if not src.exists():
        print(f"{arm}: nothing to back up"); continue
    if dst.exists():
        print(f"{arm}: backup {dst.name} already exists"); continue
    shutil.copytree(src, dst)
    print(f"{arm}: copied to {dst.relative_to(REPO)}")
OLD = {arm: pd.read_csv(COMBINED / arm / "locations.csv") for arm in ARMS if (COMBINED / arm / "locations.csv").exists()}
OLD_ROWS = {arm: sum(1 for _ in open(COMBINED / arm / "combined_hourly.csv", encoding="utf-8")) - 1 for arm in ARMS if (COMBINED / arm / "combined_hourly.csv").exists()}
print({a: f"{r:,} rows" for a, r in OLD_ROWS.items()})

from_2023_with_preinstall: backup from_2023_with_preinstall_20261119 already exists
{'from_2023_with_preinstall': '1,216,537 rows'}


## 3. Combine (cleaning rules and year ranges)

In [3]:
if not RUN:
    print("skipped (RUN = False). Set RUN = True and run this cell again.")
else:
    cmd = [sys.executable, "-m", "preprocessing.combine_dataset"] + (["--only-clean"] if ONLY_CLEAN else [])
    r = subprocess.run(cmd, cwd=MODEL, text=True, capture_output=True)
    print(r.stdout[-3000:]); print(r.stderr[-2000:])
    assert r.returncode == 0, "combine_dataset failed (see above)"

1,395,693 readings -> 1,366,342 kept
  removed     412  no_value
  removed   6,026  floor_or_negative
  removed     176  above_1000
  removed   1,528  new_year_fireworks
  removed  20,708  before_installation
  removed     501  isolated_spike
                     full: from 2021-07-09  1,366,342 rows  86 stations (85 inside the raster), pre-installation removed
                from_2023: from 2023-01-01  1,229,190 rows  86 stations (85 inside the raster), pre-installation removed
                from_2024: from 2024-01-01  1,055,689 rows  86 stations (85 inside the raster), pre-installation removed
     full_with_preinstall: from 2021-07-09  1,387,050 rows  86 stations (85 inside the raster), pre-installation kept
from_2023_with_preinstall: from 2023-01-01  1,247,786 rows  86 stations (85 inside the raster), pre-installation kept
from_2024_with_preinstall: from 2024-01-01  1,074,285 rows  86 stations (85 inside the raster), pre-installation kept




## 4. Check the result

In [4]:
for arm in ARMS:
    new_loc, old_loc = pd.read_csv(COMBINED / arm / "locations.csv"), OLD.get(arm)
    comb = pd.read_csv(COMBINED / arm / "combined_hourly.csv", usecols=["location_id", "local_timestamp"])
    rows = len(comb)
    print(f"{arm}: {rows:,} rows (was {OLD_ROWS.get(arm, 0):,}, {rows - OLD_ROWS.get(arm, 0):+,}); last reading {comb['local_timestamp'].max()}; stations {new_loc['location_id'].nunique()}")
    if old_loc is not None:
        a, b = set(old_loc["location_id"]), set(new_loc["location_id"])
        print(f"   stations kept {len(a & b)}, new {sorted(b - a)}, dropped {sorted(a - b)}")
        print("   -> SAME stations: the spatial split (and so the test stations) stays the same" if a == b
              else "   -> THE STATION SET CHANGED: the partition will be rebuilt; the test stations can move (see step 5)")

from_2023_with_preinstall: 1,247,786 rows (was 1,216,537, +31,249); last reading 2026-11-18T19:00:00+08:00; stations 86
   stations kept 85, new [4797655], dropped []
   -> THE STATION SET CHANGED: the partition will be rebuilt; the test stations can move (see step 5)


## 5. What to redo
* **Same stations, longer range:** the test stations are unchanged, but every model must be **re-trained on the longer data** to be compared with each other. Numbers from the shorter range (R2 0.72 etc.) are a different experiment: keep them as such, do not mix them in one table.
* **Station set changed:** the partition is rebuilt. To keep the old test stations, copy them from the old `processed/partition.json` (key `test`) into `split.test_stations` of `model/configs/data.yaml` (and the calibration ones into `split.calibration_stations`) before preprocessing.
* Then, in this order: `training_cells.ipynb` with `REDO = ["preprocess"]` for the first model (the Manila-only baseline), then `transfer_learning.ipynb`. In both, set `ARM` to the folder checked above.
* The source cities (`scrapers.ipynb`) should cover the same dates: `START`/`END` there match the range of the Manila data.